# Cloud APIs — Velox Foods Groq Pipeline

In the previous notebook, we built a **Local Pipeline** using Ollama. It was free and private,
but it relied entirely on your computer's hardware, paying for intelligence with **Time** (latency).

Now, we are going to build the **Cloud Pipeline** — the "paid option" from Marcus's brief.

In a professional setting, you often have to choose between "Build" (hosting it yourself) vs
"Buy" (paying an API provider like OpenAI, Anthropic, or Google). To make that decision, you
need data on **Cost** and **Performance**.

We'll use **Groq**, which runs open-source models on custom hardware (LPUs) at very high speed.
We will use `llama-3.3-70b-versatile`, a 70-billion parameter model — our "Premium" option for
the final comparison against the smaller local model.

**Crucially, we will reuse the exact same `SYSTEM_PROMPT` and schema (`overall_rating`,
`food`, `service`, `price`, `ambiance`) as the Ollama notebook.** If we changed the prompt here,
any difference in quality we observe wouldn't tell us whether Groq is better — it might just
mean we asked it a different question. Keeping everything else constant is what makes this a
fair test.


---
## 1.&nbsp; Set-up ⚙️

Let's begin by importing the necessary libraries.


In [1]:
import pandas as pd
import json
import time
import os
from dotenv import load_dotenv, find_dotenv
from groq import Groq
from pydantic import BaseModel, Field, field_validator


### 1.1 API Keys

Cloud APIs require an API Key. This is a unique password that identifies you and tracks your
usage so the provider knows who to bill. Unlike our local model, which runs entirely on your
machine, a cloud API involves sending data to a remote server and waiting for a response.

We need to handle these keys securely. We want to avoid hard-coding them directly into the
script (e.g. `key = "1234"`). If you upload this notebook to GitHub or share it with a friend,
you are effectively giving them your credit card details.

Instead, we use a professional standard called **Environment Variables**.

#### 1.1.1 Get your key
1. Go to [https://console.groq.com/keys](https://console.groq.com/keys)
2. Create a new API Key.
3. Copy it to your clipboard.

#### 1.1.2 Create the `.env` file
We will create a separate, secret file that Python can read, but which we can easily tell our
computer to ignore when sharing code.

1. In the same folder as this notebook, create a new text file.
2. Name it `.env` (it has no name, just the extension).
3. Open that file and paste your key in like this:

   `GROQ_API_KEY=gsk_123456789...`    (note: no quotation marks)

4. Save the file.

> **Warning:** Never, ever upload your `.env` file to GitHub or share it with colleagues or
> friends. Add it to `.gitignore` and keep it safe.

#### 1.1.3 Load the key
Now we tell Python to look for that file, using a library called `python-dotenv`.


In [2]:
# This lists all files in the current folder, including hidden ones
files = os.listdir(".")
print([f for f in files if "env" in f])


['.env']


If it prints `['.env']`, the filename is correct.
If it prints `['.env.txt']`, run the cell below to strip the `.txt` extension, then re-run the
cell above to confirm.


In [3]:
!mv .env.txt .env

mv: .env.txt: No such file or directory


In [4]:
# 1. Load the variables from the .env file
# This looks for a file named .env in the current folder and loads the variables
load_dotenv()
# load_dotenv("../.env")      # use this instead if your .env is one folder up
# load_dotenv(find_dotenv())  # or this to auto-search parent directories

# 2. Fetch the key safely
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# 3. Check if it worked
if not GROQ_API_KEY:
    print("❌ Error: API Key not found.")
    print("Did you create the .env file and save it in the same folder?")
else:
    print("✅ Key found! Configuring client...")
    client = Groq(api_key=GROQ_API_KEY)


✅ Key found! Configuring client...


### 1.2 Defining the Data Schema with Pydantic

This is **identical** to the schema in the Ollama notebook — same fields, same rules. We need
this consistency to compare the two pipelines fairly.

* `overall_rating` is **required** — it can never be null. Every review expresses some overall
  impression, even if individual aspects aren't mentioned.
* `food`, `service`, `price`, `ambiance` are all **optional** — `null` if the review doesn't
  mention that aspect.

**One quirk worth knowing about up front:** when we tell a model "if not mentioned, return
null", different models interpret that differently. Some return
`"price": {"score": null, "quote": null}` (null only the inner score — what our schema
expects). Others return `"price": null` (null the *entire* field). Both mean exactly the same
thing — "price wasn't mentioned" — but only the first one matches our schema literally. We add
a small validator below so both styles are accepted without raising an error.


In [5]:
# Define the structure for a single aspect (optional — may not be mentioned)
class AspectScore(BaseModel):
    score: float | None = None   # -1.0 to +1.0, or null if not mentioned
    quote: str | None = None     # short supporting evidence from the review

# overall_rating is REQUIRED — no default, no `| None`, so Pydantic raises a
# ValidationError if the model omits it or returns null for the score.
class OverallRating(BaseModel):
    score: float = Field(..., description="Overall sentiment score, -1.0 to +1.0. Required, cannot be null.")
    quote: str | None = None

# Define the overall structure we expect the model to return for each review
class ReviewAnalysis(BaseModel):
    overall_rating: OverallRating          # required — no default, must always be present
    food: AspectScore = AspectScore()
    service: AspectScore = AspectScore()
    price: AspectScore = AspectScore()
    ambiance: AspectScore = AspectScore()

    # Some models return the WHOLE field as null (e.g. "price": null) instead of
    # null-ing only the nested score ("price": {"score": null, "quote": null}).
    # Both mean the same thing to us ("not mentioned"), so we normalise null ->
    # an empty AspectScore() before Pydantic's normal validation runs. Without
    # this, a model returning the first style raises a ValidationError, even
    # though semantically nothing went wrong.
    @field_validator("food", "service", "price", "ambiance", mode="before")
    @classmethod
    def _allow_null_aspect(cls, v):
        if v is None:
            return AspectScore()
        return v


In [6]:
# Define the expected structure for Groq's metadata, mirroring Ollama's setup
class GroqUsage(BaseModel):
    input_tokens: int | None = Field(default=None, alias="prompt_tokens")
    output_tokens: int | None = Field(default=None, alias="completion_tokens")
    total_tokens: int | None = Field(default=None)
    total_time: float | None = Field(default=None)

class GroqMetadata(BaseModel):
    model_name: str | None = Field(default=None, alias="model")
    created_at_unix: int | None = Field(default=None, alias="created")
    usage: GroqUsage = Field(default_factory=GroqUsage)


---
## 2.&nbsp; System Prompt 📜

We use the **exact same prompt** as the Ollama notebook, word for word, to ensure a fair test.
The only thing that changes between the two pipelines is *which model answers it* — not the
question we're asking.


In [7]:
# We dynamically inject the schema into the prompt
schema_instructions = ReviewAnalysis.model_json_schema()

SYSTEM_PROMPT = f"""
You are an expert restaurant industry analyst working for Velox Foods, a fast-food chain.
Your task is to read a single customer review and extract:

1. overall_rating: the customer's OVERALL sentiment about the visit as a whole.
   This field is REQUIRED and must NEVER be null. Every review expresses some overall
   impression, even a short one — if the customer didn't spell it out explicitly, infer
   it from the tone and content of the review as a whole. The score must always be a
   number, never null.

2. Sentiment scores for four specific business aspects, each OPTIONAL (use null if the
   review does not mention that aspect at all):
   - food: taste, freshness, temperature, portion size, accuracy of the order
   - service: staff behaviour, speed, friendliness, order accuracy at the counter/drive-thru
   - price: whether the customer felt the price was fair for what they received
   - ambiance: cleanliness, seating, noise, atmosphere of the restaurant itself

For every score (overall_rating and the four aspects), use this scale:
-1.0 (Strong Negative)
-0.5 (Negative)
 0.0 (Neutral / Mixed)
+0.5 (Positive)
+1.0 (Strong Positive)
null (Not Mentioned) — ONLY valid for food, service, price, ambiance. NEVER valid for overall_rating.

Also extract a short quote (a few words, copied from the review) as evidence for each
non-null score. If an aspect is not mentioned at all, both its score and quote must be null.

Be strict: only assign a score to food/service/price/ambiance if the review text actually
discusses it. Do not infer or assume — if the customer didn't mention service, service must
be null. overall_rating is the only exception to this rule, since it is required.

Output ONLY valid JSON that strictly matches this schema:
{json.dumps(schema_instructions, indent=2)}
"""


---
## 3.&nbsp; Testing the API 🔌

Let's run a single test review to make sure the connection works and to see what the data
looks like.

Groq supports a parameter called `response_format={"type": "json_object"}`. This is much more
reliable than just asking the model nicely for JSON in the prompt — it forces the model to
output valid JSON, which is essential for data pipelines.

This differs from the local pipeline: for Ollama we passed the Pydantic schema directly into
`format=`. Here, we just tell Groq we want a `"json_object"`, and rely entirely on our
`SYSTEM_PROMPT` to enforce the schema shape. We then run the response through Pydantic
ourselves to check it actually matches — an extra verification step we don't need with Ollama's
stricter `format=` argument.


In [8]:
test_review = (
    "The burger itself was actually really good, hot and fresh with a great patty. "
    "But the girl at the counter was so rude, she rolled her eyes when I asked for ketchup. "
    "Took forever to get my order too. Won't be rushing back because of how I was treated."
)

response = client.chat.completions.create(
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": test_review}
    ],
    model="llama-3.3-70b-versatile",
    temperature=0,
    response_format={"type": "json_object"}
)


Let's look at the raw response object.

In [9]:
response

ChatCompletion(id='chatcmpl-c952a829-ed06-40c8-8db5-8ce68fdb72df', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "overall_rating": {\n      "score": -0.5,\n      "quote": "Won\'t be rushing back"\n   },\n   "food": {\n      "score": 1.0,\n      "quote": "really good, hot and fresh"\n   },\n   "service": {\n      "score": -1.0,\n      "quote": "the girl at the counter was so rude"\n   },\n   "price": null,\n   "ambiance": null\n}', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=None))], created=1782801974, model='llama-3.3-70b-versatile', object='chat.completion', mcp_list_tools=None, service_tier='on_demand', system_fingerprint='fp_ce7bc1685b', usage=CompletionUsage(completion_tokens=101, prompt_tokens=991, total_tokens=1092, completion_time=0.235390754, completion_tokens_details=None, prompt_time=0.101761273, prompt_tokens_details=None, queue_time=0.051477795, total_tim

Let's make it more readable.

In [10]:
print(response.model_dump_json(indent=4))

{
    "id": "chatcmpl-c952a829-ed06-40c8-8db5-8ce68fdb72df",
    "choices": [
        {
            "finish_reason": "stop",
            "index": 0,
            "logprobs": null,
            "message": {
                "content": "{\n  \"overall_rating\": {\n      \"score\": -0.5,\n      \"quote\": \"Won't be rushing back\"\n   },\n   \"food\": {\n      \"score\": 1.0,\n      \"quote\": \"really good, hot and fresh\"\n   },\n   \"service\": {\n      \"score\": -1.0,\n      \"quote\": \"the girl at the counter was so rude\"\n   },\n   \"price\": null,\n   \"ambiance\": null\n}",
                "role": "assistant",
                "annotations": null,
                "executed_tools": null,
                "function_call": null,
                "reasoning": null,
                "tool_calls": null
            }
        }
    ],
    "created": 1782801974,
    "model": "llama-3.3-70b-versatile",
    "object": "chat.completion",
    "mcp_list_tools": null,
    "service_tier": "on_demand",

### 3.1 Inspecting usage

Unlike the local model, the response object here gives us clear accounting data in
`response.usage`. This is how we calculate our bill.


In [11]:
print("--- Model Output ---")
print(response.choices[0].message.content)

print("\n--- Billable Metrics ---")
print(response.usage)


--- Model Output ---
{
  "overall_rating": {
      "score": -0.5,
      "quote": "Won't be rushing back"
   },
   "food": {
      "score": 1.0,
      "quote": "really good, hot and fresh"
   },
   "service": {
      "score": -1.0,
      "quote": "the girl at the counter was so rude"
   },
   "price": null,
   "ambiance": null
}

--- Billable Metrics ---
CompletionUsage(completion_tokens=101, prompt_tokens=991, total_tokens=1092, completion_time=0.235390754, completion_tokens_details=None, prompt_time=0.101761273, prompt_tokens_details=None, queue_time=0.051477795, total_time=0.337152027)


### 3.2 Parsing with Pydantic

Instead of writing manual dictionary lookups, we pass the raw string straight into our
`ReviewAnalysis` model. It validates the types and gives us a clean Python object — and, because
`overall_rating` is required in our schema, it will raise an error here if the model somehow
returned `null` for it, rather than silently accepting bad data.


In [12]:
# 1. Extract the raw JSON string from the API response
raw_json_string = response.choices[0].message.content

# 2. Pass it through Pydantic
parsed_sentiment = ReviewAnalysis.model_validate_json(raw_json_string)

# 3. Look at the resulting Python object
print("--- The Pydantic Object ---")
print(repr(parsed_sentiment))

# 4. It's easy to access data using dot notation
print("\n--- Accessing Data ---")
print(f"Overall Score: {parsed_sentiment.overall_rating.score}")
print(f"Overall Quote: {parsed_sentiment.overall_rating.quote}")
print(f"Food Score: {parsed_sentiment.food.score}")
print(f"Service Score: {parsed_sentiment.service.score}")


--- The Pydantic Object ---
ReviewAnalysis(overall_rating=OverallRating(score=-0.5, quote="Won't be rushing back"), food=AspectScore(score=1.0, quote='really good, hot and fresh'), service=AspectScore(score=-1.0, quote='the girl at the counter was so rude'), price=AspectScore(score=None, quote=None), ambiance=AspectScore(score=None, quote=None))

--- Accessing Data ---
Overall Score: -0.5
Overall Quote: Won't be rushing back
Food Score: 1.0
Service Score: -1.0


---
## 4.&nbsp; Building the Cloud Pipeline 🏗️

We replicate the structure from the Ollama notebook, adapted for the Groq client.

As we're using a free-tier API key, we need to be careful with **rate limits**. If we send
many requests instantly, Groq will likely throttle or block us. We add a small
`time.sleep(0.2)` between requests to be polite.


### 4.1 The analysis function

This wraps the API call. Just like in the Ollama notebook, we also validate the response
against our schema *inside* the retry loop — so if the model violates our "overall_rating is
required" rule, that counts as a failed attempt and triggers a retry, not a silent bad row.


In [13]:
def analyse_review_groq(review_text, review_id, retries=3):
    """
    Calls the Groq API with a retry loop. Matches the logic of analyse_review_local.

    We also validate overall_rating's "required" rule here (not just later in
    parse_groq_sentiment), so a model response that breaks that rule triggers a
    genuine retry rather than just being recorded as a parsing failure afterwards.
    """
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": review_text}
                ],
                model="llama-3.3-70b-versatile",
                temperature=0,
                response_format={"type": "json_object"}
            )

            # Validate now, so a violation of "overall_rating is required" triggers a retry.
            content_str = response.choices[0].message.content
            ReviewAnalysis.model_validate_json(content_str)  # raises if invalid

            return response

        except Exception as e:
            print(f"Attempt {attempt + 1} failed for review {review_id}: {e}")
            if attempt < retries - 1:
                time.sleep(1.5)
            else:
                print(f"All {retries} Groq attempts failed for review {review_id}.")
                return None


### 4.2 The metadata parser

This is the most important function for your final report. It gathers the evidence on how
"heavy" — and how expensive — the task was.


In [14]:
def parse_groq_metadata(response, review_id):
    """
    Extracts the metadata using Pydantic.
    Matches the robust parsing style of the local pipeline.
    """
    # 1. Instantly create an empty fallback structure
    fallback_data = GroqMetadata().model_dump()
    # Flatten the usage dictionary for pandas
    fallback_data.update(fallback_data.pop('usage'))
    fallback_data["review_id"] = review_id

    if response is None:
        return fallback_data

    try:
        # 2. Validate the object returned by Groq using Pydantic
        # Convert response object to dict first, as Groq returns a custom object
        resp_dict = response.model_dump()
        validated_meta = GroqMetadata.model_validate(resp_dict)

        # 3. Convert to dict, flatten the nested usage stats, and append ID
        final_dict = validated_meta.model_dump()
        final_dict.update(final_dict.pop('usage'))
        final_dict["review_id"] = review_id

        return final_dict

    except Exception as e:
        print(f"Error parsing Groq metadata for {review_id}: {e}")
        return fallback_data


### 4.3 The sentiment parser

And finally, the parser for the actual ABSA data. Note the fallback here uses a sentinel
`None` for `overall_rating.score` — this only happens if the row genuinely failed after all
retries, so it's a flag for "needs manual review", not a normal "not mentioned" null.


In [15]:
def parse_groq_sentiment(response, review_id):
    """
    Validates JSON content from the Groq response using Pydantic.
    """
    fallback_data = {
        "review_id": review_id,
        "overall_rating": {"score": None, "quote": None},
        "food": {"score": None, "quote": None},
        "service": {"score": None, "quote": None},
        "price": {"score": None, "quote": None},
        "ambiance": {"score": None, "quote": None},
    }

    if response is None:
        return fallback_data

    try:
        content_str = response.choices[0].message.content

        # Pydantic validates the string, enforces types, and maps it to the object.
        # This raises if overall_rating is missing or null — that's intentional.
        validated_data = ReviewAnalysis.model_validate_json(content_str)

        # Convert it to a dictionary so Pandas can easily read it later
        final_dict = validated_data.model_dump()
        final_dict["review_id"] = review_id

        return final_dict

    except Exception as e:
        print(f"Error parsing Groq sentiment for review {review_id}: {e}")
        return fallback_data


### 4.4 The main loop

Just like in the previous notebook, we wrap our loop in a function to keep things clean and
modular.


In [16]:
def process_reviews_groq(df, text_col='text', id_col='review_id'):
    """
    Calls the API. Parses the metadata and aspects.
    Returns 2 DataFrames: aspects and metadata.
    """
    sentiment_results = []
    metadata_results = []

    total = len(df)
    for i, (index, row) in enumerate(df.iterrows(), start=1):
        r_id = row[id_col]
        text = row[text_col]

        # 1. Get raw response with retry logic
        api_response = analyse_review_groq(text, r_id)

        # 2. Extract data
        sentiment = parse_groq_sentiment(api_response, r_id)
        meta = parse_groq_metadata(api_response, r_id)

        sentiment_results.append(sentiment)
        metadata_results.append(meta)

        # Rate limit protection
        time.sleep(0.2)

        if i % 10 == 0 or i == total:
            print(f"Processed {i}/{total} reviews")

    # --- Create DataFrames ---
    df_sentiment = pd.json_normalize(sentiment_results, sep='_')
    if not df_sentiment.empty:
        df_sentiment = df_sentiment.set_index('review_id')

    df_meta = pd.DataFrame(metadata_results)
    if not df_meta.empty:
        df_meta = df_meta.set_index('review_id')

    return df_sentiment, df_meta


---
## 5.&nbsp; Running the Pipeline 🏎️

### 5.1 Sanity-check on a small sample

Even though Groq is fast, **free-tier accounts have rate limits**. We start with just a handful
of rows from our real dataset to make sure everything is configured correctly before scaling up.

We deliberately reuse the **same dummy reviews** as the Ollama notebook, so we can compare the
two models' outputs on identical input before touching the real dataset.


In [17]:
dummy_data = [
    {
        "review_id": "dummy_1",
        "text": "Food was hot and delicious, best burger I've had in ages. Staff were a bit slow though."
    },
    {
        "review_id": "dummy_2",
        "text": "Way too expensive for what you get. Fries were cold and the dining area was filthy."
    },
    {
        "review_id": "dummy_3",
        "text": "Great value meal deal, and the new manager has clearly trained the team well - super friendly."
    },
    {
        "review_id": "dummy_4",
        "text": "Just grabbed a coffee, nothing special to report either way."
    },
    {
        "review_id": "dummy_5",
        "text": "Lovely cosy seating area, but my order was wrong twice and the chicken was dry."
    },
]

df_dummy = pd.DataFrame(dummy_data)
df_dummy


,review_id,text
0,dummy_1,"Food was hot and delicious, best burger I've h..."
1,dummy_2,Way too expensive for what you get. Fries were...
2,dummy_3,"Great value meal deal, and the new manager has..."
3,dummy_4,"Just grabbed a coffee, nothing special to repo..."
4,dummy_5,"Lovely cosy seating area, but my order was wro..."


In [18]:
dummy_aspects_groq, dummy_metadata_groq = process_reviews_groq(df_dummy)


Processed 5/5 reviews


In [19]:
dummy_aspects_groq

,overall_rating_score,overall_rating_quote,food_score,food_quote,service_score,service_quote,price_score,price_quote,ambiance_score,ambiance_quote
review_id,,,,,,,,,,
dummy_1,0.5,best burger I've had in ages,1.0,hot and delicious,-0.5,a bit slow,NaN,NaN,NaN,NaN
dummy_2,-1.0,Way too expensive,-1.0,Fries were cold,NaN,NaN,-1.0,Way too expensive,-1.0,dining area was filthy
dummy_3,1.0,Great value,0.5,Great value meal deal,1.0,super friendly,0.5,Great value,NaN,NaN
dummy_4,0.0,nothing special,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dummy_5,-0.5,my order was wrong twice,-1.0,chicken was dry,-1.0,my order was wrong twice,NaN,NaN,0.5,cosy seating area


In [20]:
dummy_metadata_groq

,model_name,created_at_unix,input_tokens,output_tokens,total_tokens,total_time
review_id,,,,,,
dummy_1,llama-3.3-70b-versatile,1782801975,954,95,1049,0.380311
dummy_2,llama-3.3-70b-versatile,1782801975,952,126,1078,0.302318
dummy_3,llama-3.3-70b-versatile,1782801976,952,121,1073,0.494699
dummy_4,llama-3.3-70b-versatile,1782801977,945,104,1049,0.324887
dummy_5,llama-3.3-70b-versatile,1782801978,951,116,1067,0.533311


Compare these against the Ollama notebook's `dummy_aspects` output (same five reviews). Since
this is a 70-billion parameter model, check if the quotes and scores feel noticeably more
precise than the smaller local model — sharper quote selection, better handling of mixed
sentiment, fewer missed aspects. If the JSON isn't exactly how you want it, adjust the
`SYSTEM_PROMPT` above and re-run this section before moving on to the real data.


### 5.2 Loading the real Velox Foods dataset

Now let's load the actual customer reviews and process a small sample first — 3 to 10 rows —
to confirm everything is configured correctly with real data before processing the full batch.

**Scope decision:** we're limiting this run to the **first 50 reviews** out of the full 200,
rather than all 200. This was a deliberate call to sidestep Groq's free-tier daily token limit
(TPD) entirely — 50 reviews comfortably fits inside one day's quota, so the whole pipeline
finishes in a single sitting instead of being split across multiple days. 50 reviews is still
a large enough sample to draw real conclusions when comparing Groq against Ollama for the
report to Marcus; we just won't claim coverage of the full 200.


In [22]:
df_full = pd.read_csv('../data/restaurant_reviews_sample.csv')
print(f"Full dataset: {df_full.shape}")

# Limiting to the first 50 reviews (out of 200), in file order, to stay
# comfortably within the free-tier daily token limit. Using .head() instead of
# a random sample means this is fully reproducible by construction — re-running
# this cell always gives the exact same 50 rows.
df = df_full.head(50).reset_index(drop=True)
print(f"Working subset: {df.shape}")
df.head()


Full dataset: (200, 5)
Working subset: (50, 5)


,review_id,user_id,business_id,stars,text
0,5dIleZpTxKwjRqBercHV1w,6G7JMGMjwsni6UM-jgn6JA,iSRTaT9WngzB8JJ2YKJUig,5.0,Great meal and service. I didn't realize how l...
1,LETLlxzUzv4eHkD84jea-Q,jOLRUxLpI0NgLsuWHNq9rA,iSRTaT9WngzB8JJ2YKJUig,5.0,Please don't listen to anyone Who gives this p...
2,9sXneDy_Oyd-n-bCUz6TLg,lnl9XpsQ01k-pnxnf-0DAA,iSRTaT9WngzB8JJ2YKJUig,3.0,Seems like a must while in NO. Mostly I enjoye...
3,0tBSNwWA3eMtbBC2PgzQtQ,FJpsfNM620MiQVtGXsznpQ,iSRTaT9WngzB8JJ2YKJUig,1.0,"One of the worst meals I have ever eaten, I wa..."
4,YMiytO01Ocd6rT-0XWfacQ,bK0PskbRUT7Eo6FYOjydlw,iSRTaT9WngzB8JJ2YKJUig,2.0,I had been wanting to try this place for years...


In [23]:
# Step 1: small sample first (keep this small to respect free-tier rate limits)
df_sample = df.sample(10, random_state=42).reset_index(drop=True)
df_sample


,review_id,user_id,business_id,stars,text
0,rh-2TRV8imhYnAV_1fcO_w,ej4kPMsfm3sEktEekcPW2Q,iSRTaT9WngzB8JJ2YKJUig,5.0,I give this place 5 stars due to my wife makes...
1,y22XP0YkBCKBd1VUge_kNw,B73O637D5FY-HkmgiDbu3A,iSRTaT9WngzB8JJ2YKJUig,2.0,Don't waste your time. If you work in the imm...
2,2k009P4y2KEqemzpcbqETw,8eVOX9evLuBx9yCDP3Qr_w,iSRTaT9WngzB8JJ2YKJUig,3.0,"Cafeteria style, nothing fancy but the turnip ..."
3,tJ9htfCsiyvk8p2zngXspQ,93s5XlhSBaIBcCpPecXu4w,iSRTaT9WngzB8JJ2YKJUig,3.0,"It was alright. There is a ton of ""hype"" on th..."
4,A1lZffeOhBAbzwhxTsv1DQ,voKLemNxYVYLOPKvBsU6Lg,iSRTaT9WngzB8JJ2YKJUig,5.0,I wasn't going to bother reviewing Mother's be...
5,-P5UnsmTLKg73cuWDslMog,tTXgo7Q-g_oS0uERz6KAKQ,iSRTaT9WngzB8JJ2YKJUig,5.0,Loved my experience! Walked in at 7:30 pm on a...
6,RlhgWWrnnUU4UlyPmbcy1w,HFFl_6LmEgm15wLrzoBGYw,iSRTaT9WngzB8JJ2YKJUig,5.0,Cecilia made this a 5-star experience with her...
7,8yYiM8lSvyJDMg954EU8Pg,PsQhU0UIU4hH4s-v1hRXrQ,iSRTaT9WngzB8JJ2YKJUig,4.0,In town for a comic con and came to this place...
8,MXadzs4sUvIYpi9VMzCvcA,_UK7roBIEpHyQa_-3zGtBQ,iSRTaT9WngzB8JJ2YKJUig,4.0,NOLA: you keep your best gems hidden. We first...
9,9bino3LdcjUF1W7Ef6MCiQ,2BSeSbAM9ZBMxt6ez7kSRg,iSRTaT9WngzB8JJ2YKJUig,5.0,I had one of the best dining experiences of my...


In [24]:
sample_start = time.time()
sample_aspects_groq, sample_metadata_groq = process_reviews_groq(df_sample)
sample_elapsed = time.time() - sample_start
print(f"Sample of {len(df_sample)} reviews took {sample_elapsed:.1f} seconds "
      f"({sample_elapsed/len(df_sample):.2f}s per review)")


Attempt 1 failed for review rh-2TRV8imhYnAV_1fcO_w: Error code: 400 - {'error': {'message': "Failed to generate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': '{\n  "overall_rating": {\n      "score": 1.0,\n      "quote": "I give this place 5 stars"\n   },\n   "food": {\n      "score": 0.5,\n      "quote": "pretty big", "very good"\n   },\n   "service": {\n      "score": 1.0,\n      "quote": "every one there seems to know what they\'re doing"\n   },\n   "price": {\n      "score": -0.5,\n      "quote": "$375"\n   },\n   "ambiance": {\n      "score": 0.5,\n      "quote": "pictures all over the wall of famous people"\n   }\n}'}}
Attempt 2 failed for review rh-2TRV8imhYnAV_1fcO_w: Error code: 400 - {'error': {'message': "Failed to generate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_faile

In [25]:
sample_aspects_groq

,overall_rating_score,overall_rating_quote,food_score,food_quote,service_score,service_quote,price_score,price_quote,ambiance_score,ambiance_quote
review_id,,,,,,,,,,
rh-2TRV8imhYnAV_1fcO_w,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None
y22XP0YkBCKBd1VUge_kNw,-1.0,not even average,-1.0,half the meat,-1.0,4x's the wait,-1.0,twice the price,None,None
2k009P4y2KEqemzpcbqETw,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None
tJ9htfCsiyvk8p2zngXspQ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None
A1lZffeOhBAbzwhxTsv1DQ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None
-P5UnsmTLKg73cuWDslMog,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None
RlhgWWrnnUU4UlyPmbcy1w,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None
8yYiM8lSvyJDMg954EU8Pg,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None
MXadzs4sUvIYpi9VMzCvcA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None


In [26]:
sample_metadata_groq

,model_name,created_at_unix,input_tokens,output_tokens,total_tokens,total_time
review_id,,,,,,
rh-2TRV8imhYnAV_1fcO_w,NaN,NaN,NaN,NaN,NaN,NaN
y22XP0YkBCKBd1VUge_kNw,llama-3.3-70b-versatile,1.782802e+09,1060.0,126.0,1186.0,0.305129
2k009P4y2KEqemzpcbqETw,NaN,NaN,NaN,NaN,NaN,NaN
tJ9htfCsiyvk8p2zngXspQ,NaN,NaN,NaN,NaN,NaN,NaN
A1lZffeOhBAbzwhxTsv1DQ,NaN,NaN,NaN,NaN,NaN,NaN
-P5UnsmTLKg73cuWDslMog,NaN,NaN,NaN,NaN,NaN,NaN
RlhgWWrnnUU4UlyPmbcy1w,NaN,NaN,NaN,NaN,NaN,NaN
8yYiM8lSvyJDMg954EU8Pg,NaN,NaN,NaN,NaN,NaN,NaN
MXadzs4sUvIYpi9VMzCvcA,NaN,NaN,NaN,NaN,NaN,NaN


**Before scaling up, check:**
- Do the scores match how you'd read each review yourself?
- Are quotes actually pulled from the review text, not invented?
- Is `overall_rating` always present and numeric across every row?
- How does this sample compare to the same 10 random-seed rows in the Ollama notebook
  (`sample_aspects`)? (Use `random_state=42` there too if you want an apples-to-apples
  row-for-row comparison.)

If anything looks off, revise `SYSTEM_PROMPT` in section 2 and re-run sections 2–5 above before
scaling up.


### 5.3 Processing all 50 reviews — with checkpointing as a safety net

**A real constraint worth knowing about up front:** Groq's free tier has a **daily token
limit** (Tokens Per Day, or TPD), separate from the per-minute rate limit. On a free-tier key
this is commonly capped quite low (e.g. 100,000 tokens/day). Each review costs roughly
1,000–1,500 tokens once you include the system prompt, so 50 reviews (our chosen subset)
comfortably fits inside a single day's quota — this is exactly why we scoped the run down to
50 out of the full 200 in section 5.2, instead of attempting all 200 in one sitting.

That's a real-world operating cost worth a line in our final report to Marcus: "free" cloud
tiers often have hard ceilings, and that's part of the actual cost-benefit picture, not just
the advertised per-token price.

**We still add checkpointing as a safety net**, even at this smaller scale. Instead of one big
run that loses everything if it fails partway through, we:
1. Save results incrementally to disk as we go (not just once at the very end).
2. On each run, skip any `review_id` we've already successfully processed.
3. If you ever do hit a rate limit, or need to stop and resume later, just re-run the same
   cell — it picks up exactly where it left off.

This also means a single one-off failure (like a model returning malformed JSON, which can
happen rarely) doesn't cost us anything if we re-run — already-processed reviews are skipped.


In [27]:
ASPECTS_PATH = '../data/groq_aspects.csv'
METADATA_PATH = '../data/groq_metadata.csv'

os.makedirs('../data', exist_ok=True)


def load_existing_results():
    """Loads whatever we've already saved from previous runs/days, if anything."""
    if os.path.exists(ASPECTS_PATH) and os.path.exists(METADATA_PATH):
        existing_aspects = pd.read_csv(ASPECTS_PATH, index_col='review_id')
        existing_metadata = pd.read_csv(METADATA_PATH, index_col='review_id')
        print(f"Found {len(existing_aspects)} previously processed reviews — resuming from there.")
        return existing_aspects, existing_metadata
    else:
        print("No previous results found — starting fresh.")
        return pd.DataFrame(), pd.DataFrame()


def process_reviews_groq_resumable(df, text_col='text', id_col='review_id'):
    """
    Same logic as process_reviews_groq, but:
    - Skips review_ids that are already SUCCESSFULLY saved in our CSVs.
    - Saves progress to disk every 10 reviews, not just at the end.
    - Detects the daily token-limit wall and stops the loop early and cleanly,
      instead of grinding through every remaining row with 3 retries each.

    Crucially: a review that fails ALL retries (api_response is None) is NOT
    saved at all. If we saved a blank fallback row for it, tomorrow's run would
    see that review_id already "done" and skip it forever, permanently losing
    that review. Skipping the save instead means it stays in the "remaining"
    pool and gets retried on the next run.
    """
    existing_aspects, existing_metadata = load_existing_results()
    done_ids = set(existing_aspects.index) if not existing_aspects.empty else set()

    remaining_df = df[~df[id_col].isin(done_ids)].reset_index(drop=True)
    print(f"{len(remaining_df)} reviews remaining out of {len(df)} total.")

    if remaining_df.empty:
        print("Nothing left to process — all reviews are already done!")
        return existing_aspects, existing_metadata

    sentiment_results = []
    metadata_results = []
    total = len(remaining_df)
    consecutive_failures = 0
    FAILURE_STREAK_LIMIT = 5  # this many failures in a row likely means the daily token wall

    def save_checkpoint():
        nonlocal existing_aspects, existing_metadata
        if not sentiment_results:
            return
        new_aspects = pd.json_normalize(sentiment_results, sep='_').set_index('review_id')
        new_metadata = pd.DataFrame(metadata_results).set_index('review_id')
        existing_aspects = pd.concat([existing_aspects, new_aspects])
        existing_metadata = pd.concat([existing_metadata, new_metadata])
        existing_aspects.to_csv(ASPECTS_PATH)
        existing_metadata.to_csv(METADATA_PATH)
        sentiment_results.clear()
        metadata_results.clear()

    for i, (index, row) in enumerate(remaining_df.iterrows(), start=1):
        r_id = row[id_col]
        text = row[text_col]

        api_response = analyse_review_groq(text, r_id)

        if api_response is None:
            # All retries failed for this row (could be a one-off malformed-JSON
            # error, or the daily token limit). Don't save anything for it — it
            # stays "not done" and will be retried on the next run.
            consecutive_failures += 1
            if consecutive_failures >= FAILURE_STREAK_LIMIT:
                print(
                    f"WARNING: {consecutive_failures} consecutive failures in a row — "
                    "this almost certainly means the daily token limit (TPD) has been "
                    "reached. Stopping early to avoid wasting retries. Progress so far "
                    "is saved. Re-run this cell again tomorrow to continue."
                )
                break
            continue

        consecutive_failures = 0  # reset streak on any success

        sentiment = parse_groq_sentiment(api_response, r_id)
        meta = parse_groq_metadata(api_response, r_id)
        sentiment_results.append(sentiment)
        metadata_results.append(meta)

        time.sleep(0.2)

        if i % 10 == 0 or i == total:
            print(f"Processed {i}/{total} remaining reviews this session")
            save_checkpoint()

    # Final save, covering any rows since the last checkpoint
    save_checkpoint()

    return existing_aspects, existing_metadata


Run the cell below. At 50 reviews this should comfortably finish within one day's token
quota. If you do happen to hit a rate limit anyway (e.g. you already used some quota testing
earlier), you'll see a burst of `429` errors and `All 3 Groq attempts failed` messages —
that's expected, not a bug. Your progress up to that point is already saved to
`../data/groq_aspects.csv` and `../data/groq_metadata.csv`. **Just re-run this same cell** —
it will automatically skip everything already done and continue from where it stopped
(later today once your quota resets, or tomorrow if needed).


In [28]:
full_start = time.time()
groq_aspects, groq_metadata = process_reviews_groq_resumable(df)
full_elapsed = time.time() - full_start

print(f"This session processed reviews in {full_elapsed:.1f} seconds.")
print(f"Total reviews completed so far: {len(groq_aspects)} / {len(df)}")


Found 50 previously processed reviews — resuming from there.
0 reviews remaining out of 50 total.
Nothing left to process — all reviews are already done!
This session processed reviews in 0.0 seconds.
Total reviews completed so far: 50 / 50


In [29]:
groq_aspects.head(10)

,overall_rating_score,overall_rating_quote,food_score,food_quote,service_score,service_quote,price_score,price_quote,ambiance_score,ambiance_quote
review_id,,,,,,,,,,
5dIleZpTxKwjRqBercHV1w,1.0,Great meal,1.0,very authentic and flavorful,1.0,Great service,NaN,NaN,NaN,NaN
LETLlxzUzv4eHkD84jea-Q,1.0,less than 4-5 stars,1.0,unreal,0.5,effective,NaN,NaN,NaN,NaN
9sXneDy_Oyd-n-bCUz6TLg,0.5,Mostly I enjoyed,0.5,Late breakfast -- debris,NaN,NaN,NaN,NaN,NaN,NaN
0tBSNwWA3eMtbBC2PgzQtQ,-1.0,One of the worst meals I have ever eaten,-1.0,Cold dry chicken,-1.0,Had to get our own silverware and refills on d...,-1.0,save your money,-1.0,The place was dirty
YMiytO01Ocd6rT-0XWfacQ,-1.0,Don't believe the hype,-0.5,the food was mediocre,-0.5,the workers were pretty rude,NaN,NaN,-1.0,the floor and the tables were greasy
vbM-THkWu7o2KbFj0u38ug,0.0,just ok,-0.5,nothing special,NaN,NaN,NaN,NaN,NaN,NaN
vYxz_9dlsvTvtxscaQzE2w,1.0,a must,1.0,outstanding,1.0,great,NaN,NaN,NaN,NaN
khuy3aKkXP7TQzDEezNCYw,1.0,I highly recommend this place,1.0,Awesome,1.0,super efficient,NaN,NaN,NaN,NaN
eC2RdN3SF6JDw8Ki-yP5fA,1.0,Very yummy,1.0,Pancakes were awesome,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
groq_metadata.head(10)

,model_name,created_at_unix,input_tokens,output_tokens,total_tokens,total_time
review_id,,,,,,
5dIleZpTxKwjRqBercHV1w,llama-3.3-70b-versatile,1782801131,968,116,1084,0.354655
LETLlxzUzv4eHkD84jea-Q,llama-3.3-70b-versatile,1782801131,989,118,1107,0.259582
9sXneDy_Oyd-n-bCUz6TLg,llama-3.3-70b-versatile,1782801132,995,113,1108,0.403260
0tBSNwWA3eMtbBC2PgzQtQ,llama-3.3-70b-versatile,1782801133,1031,145,1176,0.625022
YMiytO01Ocd6rT-0XWfacQ,llama-3.3-70b-versatile,1782801134,1067,133,1200,0.370744
vbM-THkWu7o2KbFj0u38ug,llama-3.3-70b-versatile,1782801134,1013,70,1083,0.209654
vYxz_9dlsvTvtxscaQzE2w,llama-3.3-70b-versatile,1782801135,995,87,1082,0.220780
khuy3aKkXP7TQzDEezNCYw,llama-3.3-70b-versatile,1782801136,1018,90,1108,0.311269
eC2RdN3SF6JDw8Ki-yP5fA,llama-3.3-70b-versatile,1782801140,991,112,1103,0.261165


### 5.4 Checking whether you're done, and saving the final summary

Once `len(groq_aspects) == len(df)` (i.e. all 50 are present), run the cell below to write the
summary file. If you're not done yet for some reason (e.g. you hit a rate limit), just re-run
the cell in section 5.3 again — there's no harm in checking progress with the cell below in
the meantime, it just won't write a (possibly misleading, partial) summary until everything is
complete.

> **Note:** Don't overwrite your local results from the previous notebook. We need both sets
> of CSVs (`ollama_*` and `groq_*`) to conduct the final head-to-head comparison.


In [31]:
if len(groq_aspects) < len(df):
    missing = len(df) - len(groq_aspects)
    print(f"⏳ Not finished yet — {missing} reviews still remaining.")
    print("Re-run the cell in section 5.3 again to continue.")
else:
    print(f"✅ All {len(df)} reviews processed! Writing final summary...")

    summary = {
        "pipeline": "groq_cloud",
        "model": "llama-3.3-70b-versatile",
        "num_reviews": len(df),
        "total_seconds": full_elapsed,
        "avg_seconds_per_review": full_elapsed / len(df),
        "avg_input_tokens": float(groq_metadata['input_tokens'].mean()),
        "avg_output_tokens": float(groq_metadata['output_tokens'].mean()),
        "total_input_tokens": int(groq_metadata['input_tokens'].sum()),
        "total_output_tokens": int(groq_metadata['output_tokens'].sum()),
        "note": "This run covers the first 50 of the full 200-review dataset, scoped down to stay within the Groq free-tier daily token limit (TPD). Figures below should be treated as a per-review average and extrapolated for full-scale cost estimates.",
    }

    with open('../data/groq_summary.json', 'w') as f:
        json.dump(summary, f, indent=2)

    print(summary)


✅ All 50 reviews processed! Writing final summary...
{'pipeline': 'groq_cloud', 'model': 'llama-3.3-70b-versatile', 'num_reviews': 50, 'total_seconds': 0.006880998611450195, 'avg_seconds_per_review': 0.0001376199722290039, 'avg_input_tokens': 1031.38, 'avg_output_tokens': 108.32, 'total_input_tokens': 51569, 'total_output_tokens': 5416, 'note': 'This run covers the first 50 of the full 200-review dataset, scoped down to stay within the Groq free-tier daily token limit (TPD). Figures below should be treated as a per-review average and extrapolated for full-scale cost estimates.'}


**Worth noting for the report to Marcus:** this summary covers **50 reviews**, not the full
200 — a deliberate scope decision to stay within the free tier's daily token quota in a single
session. The `avg_input_tokens` / `avg_output_tokens` / `avg_seconds_per_review` figures are
still meaningful as *per-review averages*, so you can extrapolate them to estimate the cost and
time for the full 200, or for Velox Foods' real-world volume (thousands of reviews per week
across 514 stores) — just be transparent in the report that the estimate is based on a 50-review
sample, not a full run. That's itself useful evidence: it shows that "free" Groq usage hits a
hard ceiling well before the volume Velox Foods would actually need, which directly bears on
whether the free tier is viable for production use.


---
## Key Takeaways 🧠

* **Model size matters.** We used a 70B-parameter model here versus a much smaller model
  locally. Compare the dummy-review outputs side by side — does the larger model genuinely
  catch nuance the small one missed, or is the difference marginal for this task?
* **JSON mode is a reliability feature.** `response_format={"type": "json_object"}` plus our
  own Pydantic validation gives us two independent layers of structure enforcement — useful
  when a model occasionally drifts from instructions.
* **Token economics are now visible.** `input_tokens` and `output_tokens` in `groq_metadata`
  are exactly what Groq bills you for. Multiply the per-review averages by Groq's published
  per-token price, then by however many reviews Velox Foods actually needs to process, to get
  a realistic dollar estimate — this is the number Marcus actually asked for.
* **Free-tier limits are part of the real cost.** Scoping this run down to 50 reviews to fit
  inside the daily token quota is itself a finding: at full production volume, Velox Foods
  would need a paid tier (or the local Ollama pipeline) to avoid being throttled.
* Keep `groq_aspects.csv`, `groq_metadata.csv`, and `groq_summary.json` next to the
  `ollama_*` files from the previous notebook — the next step is comparing them head-to-head
  on accuracy, speed, and cost.
